# Native CoordGen aligned depiction

**RDKit stays** as the SMILES parser and MCS / substructure mapper.
**Layout under test** is in-tree `xpict.layout2d` → Rust → static CoordGen (not RDKit's CoordGen).

Pipeline per pair `(ref, query)`:

1. Parse both with RDKit; build `Layout2DRequest` graphs.
2. Free-layout both with native CoordGen.
3. Map query→ref ring / MCS atoms; constrained-layout query onto ref's free coords.
4. Paint with `xpict._native.depict_molecule` (coords scaled into SVG `SCALE` space).
5. Compare to RDKit `GenerateDepictionMatching2DStructure`.

```bash
# from repo root
make build-python
uv run jupyter notebook notebooks/coordgen_aligned_depiction.ipynb
```

In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass

from IPython.display import HTML, SVG, display
from rdkit import Chem
from rdkit.Chem import AllChem, rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D

from xpict import _native
from xpict.contracts.scene import Scene
from xpict.draw.metrics import SCALE
from xpict.draw.svg import scene_to_svg
from xpict.layout2d import (
    Layout2DAtom,
    Layout2DBond,
    Layout2DRequest,
    Layout2DResult,
    generate_2d_coordinates,
)

assert _native.HAS_COORDGEN, "rebuild with make build-python (coordgen feature)"
rdDepictor.SetPreferCoordGen(True)
print("HAS_COORDGEN", _native.HAS_COORDGEN)
print("rdkit", Chem.rdBase.rdkitVersion)
print("SCALE", SCALE)

In [ ]:
# CoordGen free layouts use ~50 length units; paint expects bond ≈ SCALE (20).
COORDGEN_BOND = 50.0
TO_SVG = SCALE / COORDGEN_BOND


def mol_no_h(smiles: str) -> Chem.Mol:
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        raise ValueError(smiles)
    try:
        Chem.Kekulize(m, clearAromaticFlags=True)
    except Chem.KekulizeException:
        pass
    return Chem.RemoveHs(m)


def request_from_mol(
    mol: Chem.Mol,
    *,
    constrained: dict[int, tuple[float, float]] | None = None,
    fixed: dict[int, tuple[float, float]] | None = None,
    layout_flags: int = 0,
) -> Layout2DRequest:
    atoms: list[Layout2DAtom] = []
    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        kw: dict = {
            "atomic_number": atom.GetAtomicNum(),
            "formal_charge": atom.GetFormalCharge(),
        }
        if fixed and i in fixed:
            x, y = fixed[i]
            kw.update(fixed=True, x=x, y=y)
        elif constrained and i in constrained:
            x, y = constrained[i]
            kw.update(constrained=True, x=x, y=y)
        atoms.append(Layout2DAtom(**kw))
    bonds: list[Layout2DBond] = []
    for bond in mol.GetBonds():
        order = int(round(bond.GetBondTypeAsDouble())) or 1
        order = max(1, min(3, order))
        bonds.append(
            Layout2DBond(
                begin=bond.GetBeginAtomIdx(),
                end=bond.GetEndAtomIdx(),
                order=order,
            )
        )
    return Layout2DRequest(atoms=atoms, bonds=bonds, layout_flags=layout_flags)


def paint_svg(mol: Chem.Mol, layout: Layout2DResult, *, title: str = "") -> str:
    atoms = []
    for i, atom in enumerate(mol.GetAtoms()):
        sym = atom.GetSymbol()
        label = None if sym == "C" else sym
        atoms.append(
            {
                "index": i,
                "z": atom.GetAtomicNum(),
                "element": sym,
                "x": float(layout.x[i]) * TO_SVG,
                "y": float(layout.y[i]) * TO_SVG,
                "charge": int(atom.GetFormalCharge()),
                "label": label,
            }
        )
    bonds = []
    for j, bond in enumerate(mol.GetBonds()):
        bonds.append(
            {
                "index": j,
                "begin": bond.GetBeginAtomIdx(),
                "end": bond.GetEndAtomIdx(),
                "order": float(bond.GetBondTypeAsDouble() or 1.0),
            }
        )
    molecule = {"id": title or None, "atoms": atoms, "bonds": bonds}
    scene = Scene.model_validate(json.loads(_native.depict_molecule(json.dumps(molecule))))
    return scene_to_svg(scene)


def rdkit_template_svg(ref_smi: str, query_smi: str, w: int = 280, h: int = 200) -> str:
    ref = mol_no_h(ref_smi)
    query = mol_no_h(query_smi)
    AllChem.Compute2DCoords(ref)
    match = query.GetSubstructMatch(ref)
    if not match:
        # try aromatic ring SMARTS for common pairs
        ring = Chem.MolFromSmarts("a1aaaaa1")
        if ring is not None:
            rm = ref.GetSubstructMatch(ring)
            qm = query.GetSubstructMatch(ring)
            if rm and qm:
                # Build atom map query→ref for ring only via GenerateDepictionMatching2DStructure
                pass
        AllChem.Compute2DCoords(query)
    else:
        AllChem.GenerateDepictionMatching2DStructure(query, ref)
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    rdMolDraw2D.PrepareAndDrawMolecule(d, query)
    d.FinishDrawing()
    return d.GetDrawingText()


def show_row(title: str, svgs: list[tuple[str, str]]):
    parts = [f"<h3>{title}</h3><div style='display:flex;gap:1rem;flex-wrap:wrap'>"]
    for label, svg in svgs:
        parts.append(
            f"<div><div style='font:12px sans-serif;margin-bottom:4px'>{label}</div>{svg}</div>"
        )
    parts.append("</div>")
    display(HTML("".join(parts)))

In [ ]:
@dataclass(frozen=True)
class Pair:
    id: str
    ref: str
    query: str
    notes: str = ""


PAIRS = [
    Pair("phenol_anisole", "c1ccccc1O", "COc1ccccc1", "ring align"),
    Pair("benzene_toluene", "c1ccccc1", "Cc1ccccc1", "ring align"),
    Pair("naphthalene_quinoline", "c1ccc2ccccc2c1", "c1ccc2ncccc2c1", "fused"),
]


def ring_map(ref: Chem.Mol, query: Chem.Mol) -> list[tuple[int, int]]:
    """Return (query_idx, ref_idx) pairs for a shared substructure / ring."""
    # GetSubstructMatch(pattern): result[pattern_idx] = mol_idx
    match = query.GetSubstructMatch(ref)
    if match:
        return [(int(query_idx), int(ref_idx)) for ref_idx, query_idx in enumerate(match)]
    ring = Chem.MolFromSmarts("a1aaaaa1")
    assert ring is not None
    rm = ref.GetSubstructMatch(ring)
    qm = query.GetSubstructMatch(ring)
    if not rm or not qm:
        from rdkit.Chem import rdFMCS

        mcs = rdFMCS.FindMCS(
            [ref, query],
            atomCompare=rdFMCS.AtomCompare.CompareElements,
            bondCompare=rdFMCS.BondCompare.CompareOrder,
            matchValences=True,
            ringMatchesRingOnly=True,
        )
        if mcs.numAtoms < 3:
            return []
        patt = Chem.MolFromSmarts(mcs.smartsString)
        assert patt is not None
        rm = ref.GetSubstructMatch(patt)
        qm = query.GetSubstructMatch(patt)
        if not rm or not qm:
            return []
        return [(int(q), int(r)) for q, r in zip(qm, rm, strict=True)]
    return [(int(q), int(r)) for q, r in zip(qm, rm, strict=True)]


def native_aligned(ref_smi: str, query_smi: str, *, fixed: bool = False):
    ref_m = mol_no_h(ref_smi)
    query_m = mol_no_h(query_smi)
    ref_free = generate_2d_coordinates(request_from_mol(ref_m))
    query_free = generate_2d_coordinates(request_from_mol(query_m))
    pairs = ring_map(ref_m, query_m)
    seeds = {
        qi: (ref_free.x[ri], ref_free.y[ri]) for qi, ri in pairs
    }
    if fixed:
        req = request_from_mol(query_m, fixed=seeds)
    else:
        req = request_from_mol(query_m, constrained=seeds)
    query_aligned = generate_2d_coordinates(req)
    return ref_m, query_m, ref_free, query_free, query_aligned, pairs

In [ ]:
for pair in PAIRS:
    ref_m, query_m, ref_free, query_free, query_aligned, pairs = native_aligned(
        pair.ref, pair.query, fixed=False
    )
    print(f"{pair.id}: mapped {len(pairs)} atoms; constrained RMS proxy:", end=" ")
    if pairs:
        rms = math.sqrt(
            sum(
                (query_aligned.x[qi] - ref_free.x[ri]) ** 2
                + (query_aligned.y[qi] - ref_free.y[ri]) ** 2
                for qi, ri in pairs
            )
            / len(pairs)
        )
        print(f"{rms:.3f} (CoordGen units)")
    else:
        print("no map")

    show_row(
        f"{pair.id} — {pair.notes}",
        [
            ("ref free (native CoordGen)", paint_svg(ref_m, ref_free, title="ref")),
            ("query free (native CoordGen)", paint_svg(query_m, query_free, title="q-free")),
            ("query constrained (native)", paint_svg(query_m, query_aligned, title="q-align")),
            ("RDKit GenerateDepictionMatching2DStructure", rdkit_template_svg(pair.ref, pair.query)),
        ],
    )

In [ ]:
# Fixed (hard) vs constrained (soft) on phenol → anisole
ref_m, query_m, ref_free, query_free, soft, pairs = native_aligned(
    "c1ccccc1O", "COc1ccccc1", fixed=False
)
_, _, _, _, hard, _ = native_aligned("c1ccccc1O", "COc1ccccc1", fixed=True)
show_row(
    "phenol → anisole: constrained vs fixed",
    [
        ("constrained", paint_svg(query_m, soft)),
        ("fixed", paint_svg(query_m, hard)),
        ("RDKit template", rdkit_template_svg("c1ccccc1O", "COc1ccccc1")),
    ],
)